### 切分器

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text = """Pydantic AI is the Python AI SDK: a typed, extensible agent loop with every model a string swap away. The same agent runs everywhere you need it: behind a web frontend, in the terminal, on a voice call, on a durable background queue, in GitHub Actions, or as a plain object you call run() on. Image generation and embeddings come in the same box; Pydantic Graph and Pydantic Evals are separate packages, for typed control flow and for testing agent behavior the way pytest tests code.

Pydantic AI Harness has everything an agent needs for complex, long-running work, snapped on as capabilities, from memory, guardrails, and sub-agents to planning, context management, and storage, up to a complete coding agent.

Pydantic Logfire is the AI observability platform that sees your whole app, not just the LLM calls, and the Pydantic AI Gateway is one key for every model with real-time cost monitoring and budget control; the Gateway self-hosts if you would rather, and our instrumentation is plain OpenTelemetry, so any backend you already run works. Underneath both, genai-prices keeps model pricing current, and Monty is the sandboxed Python interpreter that runs model-written code.

View the complete documentation at pydantic.dev/docs/ai."""

# 定义 splitter
splitter = CharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=10,
    separator=".",
    keep_separator=True,
)
strs = splitter.split_text(text)
for str in strs:
    print(str)

#### RecursiveCharacterTextSplitter（递归字符切分）

`CharacterTextSplitter` 只认一个分隔符，容易把句子从中间切断。`RecursiveCharacterTextSplitter` 则按**分隔符优先级从粗到细递归切分**，优先在段落、换行、句子边界处切，尽量保持语义完整，是 RAG 中最常用的切分器。

默认分隔符优先级：`["\n\n", "\n", " ", ""]`（段落 → 换行 → 空格 → 单字符）。

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
    add_start_index=True,
)

chunks = recursive_splitter.split_text(text)
print("共切分为", len(chunks), "块")
for i, chunk in enumerate(chunks):
    print(f"[{i}] 长度={len(chunk)}: {chunk!r}")
documents = recursive_splitter.create_documents(chunks)
for doc in documents:
    print(doc)


**参数说明**

| 参数 | 作用 |
| --- | --- |
| `chunk_size` | 每块最大字符数 |
| `chunk_overlap` | 相邻块的重叠字符数，避免上下文被硬切断 |
| `separators` | 分隔符优先级列表，从粗到细；切不动时降级到下一级 |
| `keep_separator` | 是否把分隔符保留在块中（默认 `True`） |
| `add_start_index` | 在 metadata 中记录该块在原文的起始位置 |

In [10]:
from langchain_core.documents import Document

# chunk_size 越大块越少
for size in (200, 400):
    n = len(RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=0).split_text(text))
    print(f"chunk_size={size}, overlap=0 -> {n} 块")

# 对 Document 切分：保留原 metadata，并可补充起始位置
documents = [Document(page_content=text, metadata={"source": "pydantic_ai.txt"})]
doc_chunks = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    add_start_index=True,
).split_documents(documents)
print("split_documents ->", len(doc_chunks), "块")
print("首块 metadata:", doc_chunks[0].metadata)


chunk_size=200, overlap=0 -> 9 块
chunk_size=400, overlap=0 -> 6 块
split_documents -> 6 块
首块 metadata: {'source': 'pydantic_ai.txt', 'start_index': 0}


#### 3. MarkdownHeaderTextSplitter：按 Markdown 标题层级切分

按 `#` / `##` 等标题切分，并把标题路径写进 `metadata`，适合拆分结构化文档、便于后续按章节检索与引用。

In [11]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_text = "# 标题一\n\n正文 A。\n\n## 小节 1.1\n\n正文 B。\n\n# 标题二\n\n正文 C。"

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "H1"), ("##", "H2")]
)
for doc in markdown_splitter.split_text(markdown_text):
    print(doc.metadata, "->", doc.page_content)


{'H1': '标题一'} -> 正文 A。
{'H1': '标题一', 'H2': '小节 1.1'} -> 正文 B。
{'H1': '标题二'} -> 正文 C。


#### 4. HTMLHeaderTextSplitter：按 HTML 标题切分

按 `<h1>` / `<h2>` 等标签切分网页内容，同样把标题写入 `metadata`。

In [12]:
from langchain_text_splitters import HTMLHeaderTextSplitter

html_text = "<h1>标题一</h1><p>正文 A</p><h2>小节</h2><p>正文 B</p>"

html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=[("h1", "H1"), ("h2", "H2")])
for doc in html_splitter.split_text(html_text):
    print(doc.metadata, "->", doc.page_content)


{'H1': '标题一'} -> 标题一
{'H1': '标题一'} -> 正文 A
{'H1': '标题一', 'H2': '小节'} -> 小节
{'H1': '标题一', 'H2': '小节'} -> 正文 B


#### 5. RecursiveJsonSplitter：按 JSON 结构切分

保持 JSON 的层级结构，按 `max_chunk_size` 递归拆分较大的对象/数组，适合配置、接口返回等结构化数据。

In [14]:
from langchain_text_splitters import RecursiveJsonSplitter

json_data = {
    "name": "demo",
    "items": [{"id": 1, "text": "a" * 40}, {"id": 2, "text": "b" * 40}],
    "tags": ["x", "y", "z"],
}

json_splitter = RecursiveJsonSplitter(max_chunk_size=80)
json_chunks = json_splitter.split_json(json_data)
print("切分为", len(json_chunks), "个 JSON 片段")
print(json_chunks)


切分为 2 个 JSON 片段
[{'name': 'demo', 'items': [{'id': 1, 'text': 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa'}, {'id': 2, 'text': 'bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb'}]}, {'tags': ['x', 'y', 'z']}]


#### 6. TokenTextSplitter：按 token 数切分

以 token 为单位切分（内部用 `tiktoken`），能精确控制喂给模型的 token 预算；注意 `chunk_size` / `chunk_overlap` 的单位是 **token**。

In [16]:
from langchain_text_splitters import TokenTextSplitter

# 注意：这里的 chunk_size / chunk_overlap 单位是 token，不是字符
token_splitter = TokenTextSplitter(chunk_size=10, chunk_overlap=2, encoding_name="cl100k_base")
for chunk in token_splitter.split_text(
    "LangChain is a framework for building LLM applications with tools and agents."
):
    print(repr(chunk))


'LangChain is a framework for building LLM applications'
'LM applications with tools and agents.'


#### 7. 代码切分：from_language

`RecursiveCharacterTextSplitter.from_language(...)` 会针对语言选择合适的分隔符（函数、类、语句边界），支持 `Language.PYTHON`、`Language.JS`、`Language.MARKDOWN`、`Language.LATEX` 等，也有独立的 `PythonCodeTextSplitter`、`MarkdownTextSplitter`。

In [17]:
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

python_code = "def add(a, b):\n    return a + b\n\nclass Foo:\n    def bar(self):\n        return 42\n"

code_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=40, chunk_overlap=0
)
for chunk in code_splitter.split_text(python_code):
    print(repr(chunk))


'def add(a, b):\n    return a + b'
'class Foo:\n    def bar(self):'
'return 42'


#### 补充：需要额外依赖的切分器

以下切分器需要先安装对应依赖才能使用：

| 切分器 | 额外依赖 |
| --- | --- |
| `NLTKTextSplitter` | `pip install nltk` |
| `SpacyTextSplitter` | `spacy` + 语言模型（如 `en_core_web_sm`） |
| `SentenceTransformersTokenTextSplitter` | `sentence-transformers` |
| `KonlpyTextSplitter` | `konlpy`（韩语） |

#### 小结：两种切分器怎么选

| 切分器 | 依据 | 特点 |
| --- | --- | --- |
| `CharacterTextSplitter` | 单一分隔符 | 简单直接，但可能把句子/语义切断 |
| `RecursiveCharacterTextSplitter` | 分隔符优先级递归 | 优先在自然边界切分，尽量保持语义，RAG 首选 |
| `MarkdownHeaderTextSplitter` / `HTMLHeaderTextSplitter` | 标题层级 | 标题写入 metadata，适合结构化文档/网页 |
| `RecursiveJsonSplitter` | JSON 结构 | 结构化数据 |
| `TokenTextSplitter` | token 数 | 精确控制 token 预算 |
| `from_language`（代码切分） | 语言语法 | 源码按函数/类边界切分 |

**要点**：`chunk_size` 与 `chunk_overlap` 需要按模型上下文和语料特点调参；`split_text` 处理纯字符串，`split_documents` 处理 `Document` 并保留 metadata。